 # LDA - Scikit-learn

In [2]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity  
import numpy as np


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

df = pd.read_csv("kompas.csv")

df = df.dropna(subset=['Ringkasan'])
df = df[df['Ringkasan'].str.strip() != ""]
print(f"📦 Dataset setelah menghapus baris kosong awal: {len(df)} baris")

factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_words = set(stopwords.words('indonesian'))
exception_words = set()

def clean_kompas_pattern(text):
    if not isinstance(text, str):
        return ""
    pattern = r'^(?:[A-Z][a-zA-Z\s]*,?\s*)?kompas\.com[—\-–:]*\s*'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()
    return text

def preprocess_text(text):
    text = clean_kompas_pattern(text)  
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    stemmed_tokens = []
    for w in tokens:
        if w in exception_words:
            stemmed_tokens.append(w)
        else:
            stem = stemmer.stem(w)
            if len(stem) < 3:
                stemmed_tokens.append(w)
            else:
                stemmed_tokens.append(stem)
    return " ".join(stemmed_tokens).strip()

text_col = "Ringkasan"
df['preprocessed'] = df[text_col].astype(str).apply(preprocess_text)

before = len(df)
df = df[df['preprocessed'].str.strip() != ""]
after = len(df)

print(f"📊 Data sebelum pembersihan: {before} baris")
print(f"📉 Data sesudah pembersihan: {after} baris")
print(f"🗑️ Baris kosong yang dihapus: {before - after}")

df.to_csv("kompas_preprocessed.csv", index=False)
print("✅ File kompas_preprocessed.csv berhasil disimpan.")

print("\n🧾 Contoh hasil pembersihan:")
print(df[['Ringkasan', 'preprocessed']].head(5))


[nltk_data] Downloading package punkt to C:\Users\Mukti
[nltk_data]     Hendra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Mukti
[nltk_data]     Hendra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


📦 Dataset setelah menghapus baris kosong awal: 779 baris
📊 Data sebelum pembersihan: 779 baris
📉 Data sesudah pembersihan: 779 baris
🗑️ Baris kosong yang dihapus: 0
✅ File kompas_preprocessed.csv berhasil disimpan.

🧾 Contoh hasil pembersihan:
                                           Ringkasan  \
0  JAKARTA, KOMPAS.com– PT Sumi Rubber Indonesia ...   
1  JAKARTA, KOMPAS.com– Produsen mobil listrik as...   
2  JAKARTA, KOMPAS.com -Pilihan mobil listrik SUV...   
3  JAKARTA, KOMPAS.com– Penjualan mobil baru di I...   
4  JAKARTA, KOMPAS.com– Gubernur DKI Jakarta Pram...   

                                        preprocessed  
0  sum rubber indonesia surindo resmi hadir falke...  
1  produsen mobil listrik vietnam vinfast ramai a...  
2  pilih mobil listrik suv ringkas indonesia rama...  
3        jual mobil indonesia alami turun signifikan  
4  gubernur dki jakarta pramono anung praktik par...  


In [35]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Baca dataset
df = pd.read_csv("kompas_preprocessed.csv")
df = df[df['preprocessed'].notnull() & (df['preprocessed'].str.strip() != "")]
texts = df['preprocessed'].tolist()

# Vektorisasi teks
vectorizer = CountVectorizer(max_df=0.95, min_df=2)
X = vectorizer.fit_transform(texts)

# Tentukan jumlah topik
num_topics = 22

# Latih model LDA
lda_sklearn = LatentDirichletAllocation(
    n_components=num_topics,
    random_state=42,
    learning_method='batch'
)
lda_sklearn.fit(X)

# Tampilkan topik
feature_names = vectorizer.get_feature_names_out()
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topik {topic_idx + 1}: {' | '.join(top_words)}")

print("\n=== 🔍 TOPIK LDA (Scikit-Learn) ===")
print_topics(lda_sklearn, feature_names)



=== 🔍 TOPIK LDA (Scikit-Learn) ===
Topik 1: europa | latih | wib | calvin | verdonk | liga | lille | indonesia | posisi | jawa
Topik 2: motogp | mandalika | hasil | marquez | indonesia | sprint | race | fermin | aldeguer | marc
Topik 3: mobil | harga | bekas | milik | pasar | pilih | kendara | indonesia | konsumen | matik
Topik 4: europa | wisata | james | dean | liga | hasil | main | indonesia | gagal | tampil
Topik 5: dunia | jalan | catat | over | perintah | kota | hubung | wisatawan | emosional | kulit
Topik 6: rumah | kunjung | malaysia | zaman | tradisional | wisata | psg | masyarakat | daerah | elektronik
Topik 7: liga | inggris | lanjut | league | sabtu | premier | hasil | skor | united | chelsea
Topik 8: hitam | flek | jepang | kulit | turis | perintah | listrik | juta | agam | wajah
Topik 9: liga | dunia | italia | pekan | lawan | irak | mil | masalah | juventus | enam
Topik 10: motogp | mandalika | balap | indonesia | marquez | sirkuit | marc | tenggara | nusa | barat
Topik

In [36]:
from gensim.models.coherencemodel import CoherenceModel
from gensim import corpora
from nltk.tokenize import word_tokenize
import numpy as np

# Pastikan teks sudah dalam bentuk list token
texts = [word_tokenize(doc) for doc in df['preprocessed'].astype(str) if doc.strip() != ""]

# Buat dictionary dan corpus untuk gensim
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# Ambil topik dan kata dari model LDA Scikit-learn
n_topics = lda_sklearn.n_components
feature_names = vectorizer.get_feature_names_out()

# Konversi topik LDA sklearn ke format gensim (list of list of words)
topics = []
for topic_idx, topic in enumerate(lda_sklearn.components_):
    top_features = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]  # 10 kata teratas
    topics.append(top_features)

# Hitung Coherence Score
coherence_model = CoherenceModel(
    topics=topics,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v' 
)
coherence_score = coherence_model.get_coherence()

print(f"📈 Coherence Score (Gensim): {coherence_score:.4f}")


📈 Coherence Score (Gensim): 0.5499


In [34]:
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel
from gensim import corpora

# Konversi data untuk Gensim Coherence
tokenized_texts =  texts
id2word = corpora.Dictionary(tokenized_texts)
corpus = [id2word.doc2bow(text) for text in tokenized_texts]

results = []
for num_topics in range(7, 31):
    lda_model = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda_model.fit(X)

    # Ekstrak topik → format untuk gensim
    topics = []
    for topic_idx, topic in enumerate(lda_model.components_):
        top_words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-10 - 1:-1]]
        topics.append(top_words)

    coherence_model = CoherenceModel(
        topics=topics, 
        texts=tokenized_texts,
        dictionary=id2word, 
        coherence='c_v'
    )
    coherence = coherence_model.get_coherence()
    results.append((num_topics, coherence))
    print(f"num_topics={num_topics}, coherence={coherence:.4f}")

# Pilih hasil terbaik
best = max(results, key=lambda x: x[1])
print(f"\n✅ Jumlah topik terbaik: {best[0]} (Coherence={best[1]:.4f})")


num_topics=7, coherence=0.5217
num_topics=8, coherence=0.4555
num_topics=9, coherence=0.4647
num_topics=10, coherence=0.4846
num_topics=11, coherence=0.4883
num_topics=12, coherence=0.4877
num_topics=13, coherence=0.5014
num_topics=14, coherence=0.5158
num_topics=15, coherence=0.4505
num_topics=16, coherence=0.4532
num_topics=17, coherence=0.4894
num_topics=18, coherence=0.4962
num_topics=19, coherence=0.5150
num_topics=20, coherence=0.4298
num_topics=21, coherence=0.4679
num_topics=22, coherence=0.5499
num_topics=23, coherence=0.4671
num_topics=24, coherence=0.4495
num_topics=25, coherence=0.5368
num_topics=26, coherence=0.4920
num_topics=27, coherence=0.4910
num_topics=28, coherence=0.5043
num_topics=29, coherence=0.5136
num_topics=30, coherence=0.4457

✅ Jumlah topik terbaik: 22 (Coherence=0.5499)
